<a href="https://colab.research.google.com/github/deeksha-kn/customer_chrun-/blob/main/customer_chrun_dv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 **Data Preparation**

In [ ]:
import pandas as pd
df = pd.read_csv('/content/Telco_Customer_Churn_Dataset  (1).csv')

In [ ]:
df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [ ]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [ ]:
df = df.dropna()

In [ ]:
df.fillna(df.mean(numeric_only=True), inplace=True)

In [ ]:
df.fillna(df.mode().iloc[0], inplace=True)

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df.fillna(df.mean(numeric_only=True), inplace=True)

In [ ]:
df.drop('customerID', axis=1, inplace=True)

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns
print(cat_cols)

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'Churn'],
      dtype='object')


In [ ]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [ ]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   SeniorCitizen                          7043 non-null   int64  
 1   tenure                                 7043 non-null   int64  
 2   MonthlyCharges                         7043 non-null   float64
 3   TotalCharges                           7043 non-null   float64
 4   gender_Male                            7043 non-null   bool   
 5   Partner_Yes                            7043 non-null   bool   
 6   Dependents_Yes                         7043 non-null   bool   
 7   PhoneService_Yes                       7043 non-null   bool   
 8   MultipleLines_No phone service         7043 non-null   bool   
 9   MultipleLines_Yes                      7043 non-null   bool   
 10  InternetService_Fiber optic            7043 non-null   bool   
 11  Inte

**Split Data for Training and Testing**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df.Churn_Yes

,Churn_Yes
0,False
1,False
2,True
3,False
4,True
...,...
7038,False
7039,False
7040,False
7041,True


In [ ]:
important_features = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
    'Contract_One year',
    'Contract_Two year',
    'InternetService_Fiber optic',
    'PaymentMethod_Electronic check',
    'TechSupport_Yes',
    'OnlineSecurity_Yes'
]

X = df[important_features]
y = df['Churn_Yes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test
    random_state=42,    # reproducible results
    stratify=y          # maintain class balance
)

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)# verify split

X_train: (5634, 9)
X_test: (1409, 9)


In [ ]:
print("Training target distribution:\n", y_train.value_counts(normalize=True))
print("Testing target distribution:\n", y_test.value_counts(normalize=True))# Check distribution

Training target distribution:
 Churn_Yes
False    0.734647
True     0.265353
Name: proportion, dtype: float64
Testing target distribution:
 Churn_Yes
False    0.734564
True     0.265436
Name: proportion, dtype: float64


**Feature Selection**

In [ ]:
corr = df.corr()
corr['Churn_Yes'].sort_values(ascending=False)

,Churn_Yes
Churn_Yes,1.000000
InternetService_Fiber optic,0.308020
PaymentMethod_Electronic check,0.301919
MonthlyCharges,0.193356
PaperlessBilling_Yes,0.191825
SeniorCitizen,0.150889
StreamingTV_Yes,0.063228
StreamingMovies_Yes,0.061382
MultipleLines_Yes,0.040102
PhoneService_Yes,0.011942


In [ ]:
important_features = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
    'Contract_One year',
    'Contract_Two year',
    'InternetService_Fiber optic',
    'PaymentMethod_Electronic check',
    'TechSupport_Yes',
    'OnlineSecurity_Yes'
]

X = df[important_features]
y = df['Churn_Yes']

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression(max_iter=1000)

In [ ]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print(y_pred[:10])

[False  True False  True False  True False False False False]


**Model Selection**

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

**Model Training**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

In [ ]:
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC-AUC:", roc_auc)

Accuracy: 0.78708303761533
Precision: 0.6201298701298701
Recall: 0.5106951871657754
F1 Score: 0.5601173020527859
ROC-AUC: 0.6988258544524529


In [ ]:
print(confusion_matrix(y_test, y_pred))

[[918 117]
 [183 191]]


In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
fig = px.box(df, x='Churn_Yes', y='tenure', title="Tenure vs Churn")
fig.show()

In [ ]:
import plotly.express as px

fig = px.pie(
    df,
    names='Churn_Yes',
    title='Churn Percentage Distribution'
)
fig.show()

In [ ]:
fig = px.histogram(
    df,
    x='tenure',
    color='Churn_Yes',
    title='Tenure Distribution by Churn'
)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='tenure',
    y='MonthlyCharges',
    color='Churn_Yes',
    title='Tenure vs Monthly Charges'
)
fig.show()

In [ ]:
fig = px.sunburst(
    df,
    path=['Churn_Yes', 'Contract_One year'],
    title='Churn vs Contract Type'
)
fig.show()

In [ ]:
fig = px.bar(
    df.groupby('Churn_Yes')[['MonthlyCharges']].mean().reset_index(),
    x='Churn_Yes',
    y='MonthlyCharges',
    title='Average Monthly Charges by Churn'
)
fig.show()

In [ ]:
fig = px.area(
    df.sort_values('tenure'),
    x='tenure',
    y='MonthlyCharges',
    color='Churn_Yes',
    title='Charges Trend over Tenure'
)
fig.show()

In [ ]:
fig = px.violin(
    df,
    x='Churn_Yes',
    y='MonthlyCharges',
    box=True,
    title='Monthly Charges Distribution'
)
fig.show()

In [ ]:
import plotly.express as px

fig = px.treemap(
    df,
    path=['Churn_Yes', 'PaymentMethod_Electronic check'],
    title='Churn Distribution by Payment Method'
)
fig.show()

In [ ]:
fig = px.strip(
    df,
    x='Churn_Yes',
    y='tenure',
    title='Tenure Distribution by Churn'
)
fig.show()

In [ ]:
fig = px.density_heatmap(
    df,
    x='tenure',
    y='MonthlyCharges',
    title='Density of Customers'
)
fig.show()

In [ ]:
fig = px.ecdf(
    df,
    x='MonthlyCharges',
    color='Churn_Yes',
    title='Cumulative Distribution of Charges'
)
fig.show()